# C2 - CATSA COGNITIVE LOAD FROM HEART SIGNALS (step 11)

**Status: PRELIMINARY, public data. Estimator for one latent state (task-induced cognitive load), not the capacity label.**

Question: when a person does a demanding mental task (N-back, Stroop, logic problems, Sudoku), how do heart rate and heart rate variability change compared with the same person at rest, and can a small model tell them apart?

Data: CATSA (Mendeley Data, CC BY 4.0), 50 people, wrist Empatica E4: BVP 64 Hz, HR 1 Hz, EDA 4 Hz, TEMP 4 Hz, ACC 32 Hz. Each person has 3 minutes per condition: Baseline, Nback, Stroop, Logic, Sudoku. CATSA is used on its own and never merged with other data sets.

**Known weakness (state it in the report):** the order of the conditions is not documented in the data files, and in similar studies the baseline comes first. Baseline-vs-task may therefore be mixed with "early vs late in the session" (the same trap found in SWELL-KW). To check this, a second test compares **tasks with each other** (no baseline involved).

Rules fixed before looking at results:
1. People are never split between train and test (participant-grouped cross-validation). The participant code is never a feature.
2. Windows: 60 seconds, 30 second step (5 per condition). Labels come from the folder names, nothing invented.
3. Main target: **cognitive task (1) vs baseline (0)**. Control: pairwise task-vs-task tests.
4. Feature sets: HR/HRV only (deployable), HR/HRV + movement, + EDA and temperature (lab ceiling).
5. Features shown raw and person-normalised (own median and spread, no labels used).
6. Shuffled-label control. "Clearly above chance" only if `mean AUC > max(shuffled mean, 0.5) + 2 x shuffled std`.
7. Report accuracy, precision, recall, F1, balanced accuracy, AUC and the confusion matrix.
8. Lab tasks, not everyday life, not a medical measure.

## Cell 1 - Where the data are
Local: `data/catsa/CATSA.zip` (8.5 MB, extracted into its own folder). In Colab the cell downloads it.

In [ ]:
import os, sys, zipfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    BASE = Path('/content/catsa'); BASE.mkdir(exist_ok=True)
    ZIP = BASE / 'CATSA.zip'
    if not ZIP.exists():
        os.system(f'wget -q -O {ZIP} "https://data.mendeley.com/public-files/datasets/jwbz4ggws5/files/6174c500-e280-4d42-9096-146d6abe5245/file_downloaded"')
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = Path('/content/drive/MyDrive/AnchorAI_c2_outputs')
else:
    BASE = Path(os.environ.get('C2_CATSA_DIR', 'data/catsa'))
    ZIP = BASE / 'CATSA.zip'
    OUT_DIR = Path(os.environ.get('C2_OUT_DIR', 'outputs'))
OUT_DIR.mkdir(exist_ok=True, parents=True)
ROOT = BASE / 'extracted'
if not (ROOT / 'CATSA').exists():
    ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(ROOT)
subs = sorted((ROOT / 'CATSA').glob('Sub*'), key=lambda p: int(p.name[3:]))
print('people:', len(subs), '| first:', subs[0].name, '| last:', subs[-1].name)

## Cell 2 - Windows and features
Same beat-finding as the WESAD notebook. A window needs at least 25 valid inter-beat intervals. HRV (`rmssd`, `sdnn`, `pnn50`) comes from the beats found in the BVP signal; heart rate also comes from the `HR.csv` file (1 Hz).

In [ ]:
from scipy.signal import butter, filtfilt, find_peaks
CONDS = ['Baseline', 'Nback', 'Stroop', 'Logic', 'Sudoku']
WIN, STEP = 60, 30

def bvp_ibi(x):
    b, a = butter(3, [0.5 / 32, 8 / 32], btype='band')
    y = filtfilt(b, a, x)
    pk, _ = find_peaks(y, distance=int(64 * 0.33), prominence=0.4 * np.std(y))
    ibi = np.diff(pk / 64.0)
    return ibi[(ibi >= 0.33) & (ibi <= 1.5)]

def read_col(p):
    d = pd.read_csv(p)
    return d.apply(pd.to_numeric, errors='coerce').dropna().to_numpy()

rows = []
for sdir in subs:
    sid = sdir.name
    for c in CONDS:
        cdir = sdir / c
        try:
            bvp = read_col(cdir / 'BVP.csv').ravel(); hr = read_col(cdir / 'HR.csv').ravel()
            eda = read_col(cdir / 'EDA.csv').ravel(); tmp = read_col(cdir / 'TEMP.csv').ravel(); acc = read_col(cdir / 'ACC.csv')
        except Exception as e:
            print('skip', sid, c, e); continue
        n_sec = int(len(bvp) / 64)
        for start in range(0, n_sec - WIN + 1, STEP):
            ibi = bvp_ibi(bvp[start * 64:(start + WIN) * 64])
            if len(ibi) < 25:
                continue
            d = np.diff(ibi)
            a = acc[start * 32:(start + WIN) * 32]
            mag = np.sqrt((a.astype(float) ** 2).sum(axis=1)) if a.ndim == 2 else np.abs(a).ravel()
            e = eda[start * 4:(start + WIN) * 4]
            rows.append({'subject': sid, 'cond': c, 'start_s': start,
                         'hr_mean': 60.0 / ibi.mean(), 'hr_file': hr[start:start + WIN].mean(), 'hr_std': np.std(60.0 / ibi),
                         'rmssd': np.sqrt(np.mean(d ** 2)) * 1000, 'sdnn': ibi.std() * 1000, 'pnn50': np.mean(np.abs(d) > 0.05) * 100,
                         'acc_mean': mag.mean(), 'acc_std': mag.std(),
                         'eda_mean': e.mean(), 'eda_slope': np.polyfit(np.arange(len(e)), e, 1)[0], 'temp_mean': tmp[start * 4:(start + WIN) * 4].mean()})
W = pd.DataFrame(rows)
W['y_load'] = (W['cond'] != 'Baseline').astype(int)
W.to_csv(OUT_DIR / 'catsa_wrist_windows.csv', index=False)
print('windows:', len(W), '| people:', W['subject'].nunique())
print(W['cond'].value_counts().reindex(CONDS).to_string())

## Cell 3 - What changes with a demanding task
Person-normalised values (0 = that person's usual), averaged by condition.

In [ ]:
HRV = ['hr_mean', 'hr_std', 'rmssd', 'sdnn', 'pnn50']
MOVE = ['acc_mean', 'acc_std']
LAB = ['eda_mean', 'eda_slope', 'temp_mean']
ALLF = HRV + MOVE + LAB

def person_norm(df, cols):
    out = df.copy()
    for c in cols:
        g = df.groupby('subject')[c]
        med = g.transform('median'); sp = g.transform(lambda s: max(s.quantile(0.75) - s.quantile(0.25), 1e-6))
        out[c] = (df[c] - med) / sp
    return out
WN = person_norm(W, ALLF)
print('Mean person-normalised value per condition (0 = usual):')
print(WN.groupby('cond')[['hr_mean', 'rmssd', 'sdnn', 'pnn50', 'acc_mean', 'eda_mean']].mean().reindex(CONDS).round(2).to_string())
print('\nRaw means per condition (bpm / ms):')
print(W.groupby('cond')[['hr_mean', 'rmssd', 'sdnn']].mean().reindex(CONDS).round(1).to_string())

## Cell 4 - Cross-validation helper
Participant-grouped, 10 different fold layouts, out-of-fold predictions pooled per layout, shuffled-label control.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score, roc_auc_score, confusion_matrix

def make_model(name):
    if name == 'Majority':
        return DummyClassifier(strategy='prior')
    if name == 'LR':
        return Pipeline([('imp', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
                         ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))])
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('clf', RandomForestClassifier(n_estimators=150, min_samples_leaf=5, class_weight='balanced', random_state=42, n_jobs=-1))])

REPEATS = 10
def run_cv(data, cols, label, model_name, shuffled=False):
    X, y, g = data[cols].to_numpy(), data[label].to_numpy(), data['subject'].to_numpy()
    rows, cms = [], []
    for s in range(REPEATS):
        yy = np.random.RandomState(900 + s).permutation(y) if shuffled else y
        prob = np.full(len(yy), np.nan)
        for tr, te in StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=s).split(X, yy, g):
            if len(np.unique(yy[tr])) < 2:
                prob[te] = yy[tr].mean(); continue
            prob[te] = make_model(model_name).fit(X[tr], yy[tr]).predict_proba(X[te])[:, 1]
        pred = (prob >= 0.5).astype(int)
        auc = 0.5 if model_name == 'Majority' else roc_auc_score(yy, prob)
        rows.append([accuracy_score(yy, pred), precision_score(yy, pred, zero_division=0), recall_score(yy, pred, zero_division=0),
                     f1_score(yy, pred, zero_division=0), balanced_accuracy_score(yy, pred), auc])
        cms.append(confusion_matrix(yy, pred, labels=[0, 1]))
    return np.array(rows), np.mean(cms, axis=0)

RES, CMS = [], {}
def evaluate(exp, data, cols, label, model_name):
    real, cm = run_cv(data, cols, label, model_name)
    shuf = real if model_name == 'Majority' else run_cv(data, cols, label, model_name, shuffled=True)[0]
    thr = max(shuf[:, 5].mean(), 0.5) + 2 * shuf[:, 5].std()
    RES.append({'experiment': exp, 'model': model_name, 'rows': len(data), 'people': data['subject'].nunique(),
                'accuracy': real[:, 0].mean(), 'precision': real[:, 1].mean(), 'recall': real[:, 2].mean(), 'F1': real[:, 3].mean(),
                'balanced acc': real[:, 4].mean(), 'AUC': real[:, 5].mean(), 'AUC std': real[:, 5].std(), 'shuffled AUC': shuf[:, 5].mean(),
                'clearly above chance?': 'n/a' if model_name == 'Majority' else ('YES' if real[:, 5].mean() > thr else 'no')})
    CMS[(exp, model_name)] = cm
print('helper ready')

## Cell 5 - Experiments
Main: task vs baseline. Control: task vs task, with no baseline involved. If baseline-vs-task is high but task-vs-task is at chance, the strong score may partly reflect session order or simply "any mental effort", and not a particular task.

In [ ]:
SETS = [
    ('C1 HR/HRV, raw', W, HRV),
    ('C2 HR/HRV, person-normalised', WN, HRV),
    ('C3 HR/HRV + movement, person-normalised', WN, HRV + MOVE),
    ('C4 + EDA and temperature (lab ceiling), person-normalised', WN, ALLF),
]
for name, data, cols in SETS:
    for m in ['Majority', 'LR', 'RF']:
        evaluate(name, data, cols, 'y_load', m)
import itertools
TASKS = ['Nback', 'Stroop', 'Logic', 'Sudoku']
for a, b in itertools.combinations(TASKS, 2):
    sub = WN[WN['cond'].isin([a, b])].copy(); sub['y_t'] = (sub['cond'] == b).astype(int)
    for m in ['Majority', 'RF']:
        evaluate(f'T {a} vs {b}, HR/HRV person-normalised (control)', sub, HRV, 'y_t', m)
res = pd.DataFrame(RES)
show = ['experiment', 'model', 'rows', 'people', 'accuracy', 'precision', 'recall', 'F1', 'balanced acc', 'AUC', 'shuffled AUC', 'clearly above chance?']
print(res[show].round(3).to_string(index=False))
res.round(4).to_csv(OUT_DIR / 'catsa_load_results.csv', index=False)

## Cell 6 - Confusion matrix and figure

In [ ]:
import matplotlib.pyplot as plt
main = res[(res['experiment'].str.startswith('C2')) & (res['model'] != 'Majority')].sort_values('AUC', ascending=False).iloc[0]
cm = CMS[(main['experiment'], main['model'])]
print(f"{main['experiment']} / {main['model']}: AUC {main['AUC']:.2f}, balanced acc {main['balanced acc']:.2f}, F1 {main['F1']:.2f}")
print(pd.DataFrame(cm.round(0), index=['true baseline', 'true task'], columns=['pred baseline', 'pred task']).to_string())
rf = res[res['model'] == 'RF'].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(9, 5)); y = np.arange(len(rf))
ax.barh(y, rf['AUC'], xerr=rf['AUC std'], color='tab:blue', alpha=0.85, label='real labels')
ax.scatter(rf['shuffled AUC'], y, color='k', zorder=3, label='shuffled labels')
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1)
ax.set_yticks(y); ax.set_yticklabels([e[:60] for e in rf['experiment']], fontsize=7); ax.invert_yaxis()
ax.set_xlim(0.3, 1.0); ax.set_xlabel('ROC-AUC (Random Forest, people held out)'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT_DIR / 'catsa_load_results.png', dpi=120); plt.show()

## Cell 7 - How to read this (report text)
- **C2 vs C1:** if person-normalised beats raw, the personal-baseline idea is supported on these data.
- **Task-vs-task control:** near chance means the heart signals show "mental effort" in general, not which task. High baseline-vs-task with chance task-vs-task is still consistent with a real load effect, but the baseline-first order cannot be ruled out because the data do not document the order.
- **C4 is a ceiling**, not deployable (phones and most watches give no EDA).
- **Limits to state**: lab tasks of 3 minutes, 50 people, one wearable, unknown condition order, no everyday life. The output is a **cognitive load estimate**, not a diagnosis.